In [1]:
import cv2
import einops
import matplotlib.pyplot as plt
import mediapy
import numpy as np

import jax.numpy as jnp

from openpi.policies.libero_reason_dataset import LiberoSkillReasonDataset
from openpi.training import config as _config

In [2]:
data_config = _config.get_config('pi05_libero_skill_reason_fixed')
dataset = LiberoSkillReasonDataset(data_config.data.base_config, data_config.model.action_horizon)

The dataset you requested (None) is in 2.0 format.
While current version of LeRobot is backward-compatible with it, the version of your dataset still uses global
stats instead of per-episode stats. Update your dataset stats to the new format using this command:
```
python lerobot/common/datasets/v21/convert_dataset_v20_to_v21.py --repo-id=None
```

If you encounter a problem, contact LeRobot maintainers on [Discord](https://discord.com/invite/s3KuuzsPFb)
or open an [issue on GitHub](https://github.com/huggingface/lerobot/issues/new/choose).



Resolving data files:   0%|          | 0/4338 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/163 [00:00<?, ?it/s]

Using new skill reasoning dataset


In [3]:
import os
from pathlib import Path
import sys
SCRIPT_DIR = Path("../py_script")
sys.path.append(str(SCRIPT_DIR))
from vlm_interfaces import *

In [21]:
from vla_verify.scene_graph import TaskSceneGraph
from vla_verify.verifier import VLAVerifier
PDDL_PATH = SCRIPT_DIR / "pddl" / "pick_place_domain.pddl"
pddl_domain_text = open(PDDL_PATH).read()

llm_interface, vlm_interface = get_openrouter_interfaces()
scene_graph = TaskSceneGraph(pddl_domain_text, vlm_interface)
verifier = VLAVerifier(scene_graph, llm_interface)

Using OpenRouter
  LLM: google/gemini-3-flash-preview
  VLM: google/gemini-3.1-pro-preview


In [22]:
def image_tensor_to_cv2(image, resolution=(512,512)):
    return cv2.resize(np.array(einops.rearrange(image, "c h w -> h w c") * 255, dtype=np.uint8), resolution, interpolation=cv2.INTER_LANCZOS4)

def get_episode(episode_idx):
    reasonings = dataset.reasoning[episode_idx]
    start_idx = dataset.episode_starts[episode_idx]
    end_idx = dataset.episode_ends[episode_idx]
    data = dataset.hf_dataset[int(start_idx)]
    video_frames = []
    for i in range(start_idx, end_idx):
        img_data = dataset.hf_dataset[i]['image']
        video_frames.append(image_tensor_to_cv2(img_data))
    return reasonings, video_frames

reasonings, video_frames = get_episode(2)
mediapy.write_video(f'sample.mp4', video_frames, fps=20)

/tmp/ipykernel_138424/4193395926.py:2: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  return cv2.resize(np.array(einops.rearrange(image, "c h w -> h w c") * 255, dtype=np.uint8), resolution, interpolation=cv2.INTER_LANCZOS4)


In [23]:
def _nl_task_to_pddl(llm_response, avail_actions):
    pass

def process_episode(episode):
    reasonings, video_frames = episode
    task = reasonings['segments'][0]['instruction'].split(':', 1)[-1].strip()
    print(task)
    #task = "stack the two black bowls."
    scene_graph.read_image(video_frames, hint=f"The robot is trying to {task}", ground=True)
    results = scene_graph.ground_video(additional_points_labels=[
        ("robot", [[255, 90]])
    ])
    for segment in reasonings['segments'][1:]:
        print(segment['skill'])
        pass


In [24]:
%env CC=/usr/bin/gcc
process_episode((reasonings, video_frames))

env: CC=/usr/bin/gcc
put both the cream cheese box and the butter in the basket
  [VLM] Querying VLM for scene graph construction...


INFO 2026-03-18 17:19:12,962 138424 sam3_video_predictor.py: 302: using the following GPU IDs: [0]
INFO 2026-03-18 17:19:12,963 138424 sam3_video_predictor.py: 318: 


	*** START loading model on all ranks ***


INFO 2026-03-18 17:19:12,963 138424 sam3_video_predictor.py: 320: loading model on rank=0 with world_size=1 -- this could take a while ...


  [VLM] Time elapsed: 32.314298641402274
raw_pddl_state (define (problem tabletop_problem)
 (:domain tabletop)
 (:objects
  robot_0 - robot
  table_0 - immovable ; dark brown round table | background
  basket_0 - graspable ; white woven basket | left side of the table, foreground
  soup_can_0 - graspable ; red and green can | middle left of the table, middle
  butter_0 - graspable ; small orange rectangular box | middle left of the table, foreground
  milk_carton_0 - graspable ; red and white milk carton | middle of the table, foreground
  syrup_bottle_0 - graspable ; orange bottle | right side of the table, background
  blue_can_0 - graspable ; blue can | right side of the table, middle
  juice_carton_0 - graspable ; orange juice carton | right side of the table, middle
  cream_cheese_box_0 - graspable ; small blue rectangular box | right side of the table, foreground
 )
 (:init
  (free robot_0)
  (on basket_0 table_0)
  (open basket_0)
  (on soup_can_0 table_0)
  (on butter_0 table_0

INFO 2026-03-18 17:19:30,941 138424 sam3_video_base.py: 125: setting max_num_objects=10000 and num_obj_for_compile=16
INFO 2026-03-18 17:19:32,053 138424 sam3_video_predictor.py: 322: loading model on rank=0 with world_size=1 -- DONE locally
INFO 2026-03-18 17:19:32,053 138424 sam3_video_predictor.py: 333: 


	*** DONE loading model on all ranks ***




Grounding objects:
0 a dark brown round table background
1 a white woven basket left side of the table, foreground
2 a red and green can middle left of the table, middle
3 a small orange rectangular box middle left of the table, foreground
4 a red and white milk carton middle of the table, foreground
5 a orange bottle right side of the table, background
6 a blue can right side of the table, middle
7 a orange juice carton right side of the table, middle
8 a small blue rectangular box right side of the table, foreground
9 tracks.
9 objects.
Matching: {5: 'orange_juice_carton_right_side_of_the_table,_middle_3', 2: 'orange_juice_carton_right_side_of_the_table,_middle_1', 4: 'orange_juice_carton_right_side_of_the_table,_middle_4', 3: 'orange_juice_carton_right_side_of_the_table,_middle_2', 6: 'orange_juice_carton_right_side_of_the_table,_middle_6', 7: 'orange_juice_carton_right_side_of_the_table,_middle_0', 8: 'orange_juice_carton_right_side_of_the_table,_middle_5', 0: 'dark_brown_round_tab

...

  0%|          | 0/255 [00:00<?, ?it/s]

0it [00:00, ?it/s]

Done.


/work/11430/jpeng303/vista/workspace/mujoco_test/thirdparty/sam3/sam3/visualization_utils.py:185: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=figsize)


PICKUP_FROM(butter, table)
PLACE_IN(butter, basket)
PLACE_IN(butter, basket)
PICKUP_FROM(cream cheese box, table)
PICKUP_FROM(cream cheese box, table)
PLACE_IN(cream cheese box, basket)
PLACE_IN(cream cheese box, basket)


In [ ]:
print(scene_graph.summary())
import torch
print(torch.__version__)

In [ ]:
import inspect
print(inspect.getsource(scene_graph._read_image))

Using Python 3.12.12 environment at: /work/11430/jpeng303/vista/workspace/mujoco_test/mujoco_playground/.venv
Resolved 31 packages in 1.73s                                        
Prepared 31 packages in 30.49s                                           
Uninstalled 25 packages in 8.43s
Installed 31 packages in 3.50s                              
 + cuda-bindings==12.9.4
 + cuda-pathfinder==1.2.2
 - filelock==3.20.2
 + filelock==3.20.0
 - fsspec==2025.3.0
 + fsspec==2025.12.0
 ~ jinja2==3.1.6
 - markupsafe==3.0.3
 + markupsafe==3.0.2
 ~ mpmath==1.3.0
 ~ networkx==3.6.1
 - numpy==1.26.4
 + numpy==2.3.5
 - nvidia-cublas-cu12==12.9.1.4
 + nvidia-cublas-cu12==12.6.4.1
 - nvidia-cuda-cupti-cu12==12.9.79
 + nvidia-cuda-cupti-cu12==12.6.80
 - nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-nvrtc-cu12==12.6.77
 - nvidia-cuda-runtime-cu12==12.9.79
 + nvidia-cuda-runtime-cu12==12.6.77
 - nvidia-cudnn-cu12==9.20.0.48
 + nvidia-cudnn-cu12==9.10.2.21
 - nvidia-cufft-cu12==11.4.1.4
 + nvidia-cufft-cu1